<a href="https://colab.research.google.com/github/Blarg2134/MuhammudElsell_INFO4670_Fall2026/blob/main/Week5_INFO4670_Assignment2_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

In [125]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Setup — load the three files (given)

In [126]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [127]:
import pandas as pd
student_records = pd.read_csv('student_records.csv')
n_missing_study = student_records['study_hours_reported'].isna().sum()

**Your justification (1–2 sentences):** _..._

In [128]:
# I would use imputation myself by either using the mean/median or ignoring the missing value lines considering it accounts for 255 missing
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [129]:
students = student_records
students["housing_clean"] = (
    students["housing"]
    .str.strip()
    .str.lower()
    .replace({
        r".*on.*": "On Campus",
        r".*off.*|.*parent.*|.*rent.*": "Off Campus",
        r".*own.*": "Own"
    }, regex=True)
)


In [130]:
# since there's inconsistent formatting I used .replace to account for the different formats to put them into the three different group.
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off Campus': 945, 'On Campus': 590, 'with family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [131]:
import numpy as np
impossible_ages_filter = (students['age'] < 0) | (students['age'] > 120)
impossible_ages = sorted(students.loc[impossible_ages_filter, 'age'].unique())
n_neg_work = (students['work_hours_per_week'] < 0).sum()

In [132]:
# I removed the impossible values for human age ranges, by factoring out anything above 120 or below 0, while also getting rid of any negative work hour values
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [133]:
students_dedup = students.drop_duplicates(subset=['student_id'])

**Why not de-dupe enroll / activity? (1 sentence):** _..._

In [134]:
# you can't de-dupe enrollement or activity logs because the tables themselves have multiple rows per student assigned that are different course enrollements and weekly activities.
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [135]:
import pandas as pd
weekly_activity = pd.read_csv('weekly_activity.csv')
weekly_activity['student_id'] = (
    weekly_activity['student_id']
    .astype(str)
    .str.replace("NU-", "", regex=False)
    .astype(int)
)

students_dedup.loc[:,'student_id'] = (
    students_dedup['student_id']
    .astype(str)
    .str.replace("NU-", "", regex=False)
    .astype(int)
)
activity_summary = weekly_activity.groupby('student_id')['minutes_active'].sum().reset_index()
analysis = students_dedup.merge(activity_summary, on='student_id', how='left')

In [136]:
# I had to standardize the formatting so records could be put together. The different logs merges the different activity records into one row per student
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [137]:
enrollment = pd.read_csv('course_enrollments.csv')
valid_student_ids = set(analysis['student_id'])
enroll_sids = enrollment['sid'].astype(str).str.replace("NU-", "", regex=False).astype(int)
matched = int(enroll_sids.isin(valid_student_ids).sum())

In [138]:
# using .isin shows that 14 orphan IDs are present of students are not present in the primary data set.
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [139]:
import pandas as pd
enrollment = pd.read_csv('student_records.csv')
dates_parsed = pd.to_datetime(enrollment['enrollment_date'], format="mixed", errors="coerce")

In [140]:
# I had to import and define enrollment first. The dates change where they sometimes follow a standard format as mm-dd-yyyy but yyyy-dd-mm would read differently if not accounted for.
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [141]:
analysis["study_z"] = (analysis["study_hours_reported"] - analysis["study_hours_reported"].mean()) / analysis["study_hours_reported"].std()
analysis["gpa_band"] = pd.cut(analysis["final_gpa"], bins=[-0.01, 1, 2, 3, 4])

In [142]:
# Z score rescales the study activity to have a mean of 0 and groups final_gpa with pd.cut into 4 GPA bands.
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [143]:
import os
analysis.to_csv("northgate_clean.csv", index=False)

In [144]:
# using index=False makes sure that it doesn't overwrite the raw files and saves the cleaned version without adding extra index row columns.
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**

-I would use imputation myself by either using the mean/median or ignoring the missing value lines considering it accounts for 255 missing

-Since there's inconsistent formatting I used .replace to account for the different formats to put them into the three different group.

-I removed the impossible values for human age ranges, by factoring out anything above 120 or below 0, while also getting rid of any negative work hour values

-You can't de-dupe enrollement or activity logs because the tables themselves have multiple rows per student assigned that are different course enrollements and weekly activities.

-I had to standardize the formatting so records could be put together. The different logs merges the different activity records into one row per student

-Using .isin shows that 14 orphan IDs are present of students are not present in the primary data set.

- I had to import and define enrollment first. The dates change where they sometimes follow a standard format as mm-dd-yyyy but yyyy-dd-mm would read differently if not accounted for.

-Z score rescales the study activity to have a mean of 0 and groups final_gpa with pd.cut into 4 GPA bands.
-Using index=False makes sure that it doesn't overwrite the raw files and saves the cleaned version without adding extra index row columns.

-Cleaning the dataset helps removes the invalid outliers or the duplicates. This shows a positive correlation between study time and gpa.




### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [145]:
mean_gpa = analysis["final_gpa"].mean()
print(f"Mean GPA (clean): {mean_gpa:.2f}")
print("\nMean Study Z-Score by GPA Band:")
print(analysis.groupby("gpa_band", observed=False)["study_z"].mean())

Mean GPA (clean): 2.31

Mean Study Z-Score by GPA Band:
gpa_band
(-0.01, 1.0]   -1.163869
(1.0, 2.0]     -0.944451
(2.0, 3.0]      0.137437
(3.0, 4.0]      1.037104
Name: study_z, dtype: float64


In [146]:
# Cleaning the dataset helps removes the invalid outliers or the duplicates. This shows a positive correlation between study time and gpa.
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
